# Follow-up: leak_v2, second seeds for step 4a, longer training

Trains five models (`configs/followup_v1.yaml`), then scores them on when to guess together with the
predictions saved by step 4a, and repeats the step 4b hierarchy scoring on the leak_v2 GRU.
Results are on validation data; the test split is not opened. Every comparison was fixed beforehand.

1. **leak_v2**: `gru_uniform` (2 seeds) is the step 3 GRU baseline with the leak audit's uniform resampling.
2. **Second seeds**: `cnn_anytime_s43`, `gru_no_consistency_s43`, with exactly the step 4a settings.
3. **Exploratory**: `gru_anytime_long_s42`, the step 4a method trained for twice as many steps.

**Before running:**
1. In **Input**, keep the `AI_pictionary` output of the **step 4a version** (it holds
   `runs/anytime_v1/anytime/*.npz`), as for step 4b.
2. Keep your Kaggle Dataset attached and **Internet on**.
3. Choose **GPU T4 x2**, then **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping before using GPU quota")

found = [p.parent for p in Path("/kaggle/input").rglob("gru_prefix_s42.npz")]
if not found:
    raise RuntimeError("step 4a predictions not found: attach the step 4a version of the AI_pictionary output")
OLD_PREDICTIONS = found[0]
print("step 4a predictions:", OLD_PREDICTIONS, sorted(p.name for p in OLD_PREDICTIONS.glob("*.npz")))


In [ ]:
MAX_MINUTES = 300
TOP = Path("/kaggle/working/runs/followup_v1")
!PYTHONPATH=src python -m quickdraw_exp.experiments --matrix configs/followup_v1.yaml --max-minutes {MAX_MINUTES}
# Checkpoint predictions of the new models (GPU), saved as <run>.npz.
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/followup_v1.yaml --runs {TOP} --out {TOP}/predictions --num-workers 4


In [ ]:
# All predictions in one folder: step 4a's and the new ones. Then score without running any model.
ALL = Path("/kaggle/working/runs/predictions_all")
ALL.mkdir(parents=True, exist_ok=True)
for folder in (OLD_PREDICTIONS, TOP / "predictions"):
    for p in folder.glob("*.npz"):
        target = ALL / p.name
        if not target.exists():
            shutil.copy(p, target)
print(sorted(p.name for p in ALL.glob("*.npz")))
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/followup_v1.yaml --predictions {ALL} --out /kaggle/working/runs/followup_v1_anytime
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/anytime_v1.yaml --predictions {ALL} --out /kaggle/working/runs/anytime_v1_two_seeds
!PYTHONPATH=src python -m quickdraw_exp.hierarchy --matrix configs/hierarchy_eval_followup_v1.yaml --predictions {ALL}


In [ ]:
import json

def show(rows, label):
    print(f"\n{label}")
    for row in rows:
        tag = f"[{row['precision_target']}] " if "precision_target" in row else ""
        if row["difference"] is None:
            print(f"  {tag}{row['name']}: not available")
        else:
            print(f"  {tag}{row['name']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, difference {row['difference']:+.4f}  "
                  f"(a {[round(v, 4) for v in row['a_values']]}, b {[round(v, 4) for v in row['b_values']]})")

training = json.loads((TOP / "results.json").read_text())
print("training", "COMPLETE" if training["complete"] else "INCOMPLETE, missing: " + ", ".join(training["missing"]))

# leak_v2 rule (configs/followup_v1.yaml): top-1 by ink fraction against the step 3 GRU baseline.
leak_v1 = json.loads(Path("reports/leak_v1/results.json").read_text())["table"]["gru_prefix"]
uniform = training["table"].get("gru_uniform")
print("\nTop-1 by ink fraction, val 1000 per class (gru_prefix from reports/leak_v1):")
print("  gru_prefix  ", {f: round(v["top1"]["mean"], 4) for f, v in leak_v1["by_frac"].items()},
      "early", round(leak_v1["early_top1"]["mean"], 4))
if uniform:
    print("  gru_uniform ", {f: round(v["top1"]["mean"], 4) for f, v in uniform["by_frac"].items()},
          "early", round(uniform["early_top1"]["mean"], 4), "seeds", uniform["seeds"])
    print(f"  early top-1 uniform - prefix: {uniform['early_top1']['mean'] - leak_v1['early_top1']['mean']:+.4f}")
for name, row in training["table"].items():
    print(f"  {name:20s} seeds {row['seeds']} top-1 at full drawing {row['by_frac']['1.0']['top1']['mean']:.4f}")

followup = json.loads(Path("/kaggle/working/runs/followup_v1_anytime/results.json").read_text())
print("\nmissing conditions:", followup["missing_conditions"])
show(followup["comparisons"], "Follow-up comparisons (AUC, evaluation half):")
two = json.loads(Path("/kaggle/working/runs/anytime_v1_two_seeds/results.json").read_text())
show(two["comparisons"], "Step 4a comparisons with two seeds everywhere (AUC, evaluation half):")
hier = json.loads(Path("/kaggle/working/runs/hierarchy_followup_v1/results.json").read_text())
print("\nmissing hierarchy runs:", hier["missing_runs"])
show(hier["comparisons"], "Step 4b on the leak_v2 GRU (reward area, evaluation half):")

print("\nRisk control and AUC per condition (all runs):")
for condition, policies in {**two["table"], **followup["table"]}.items():
    for policy, e in policies.items():
        if "earliness_at_gap_0.02" in e:
            print(f"  {condition:20s} {policy:7s} seeds {e['seeds']} auc {e['auc']['mean']:.4f}  earliness@gap5% "
                  f"{e['earliness_at_gap_0.05']['mean']:.3f} met {e['met_gap_0.05']}")
for name in training["missing"]:
    log = TOP / f"{name}.log"
    if log.exists():
        print(f"--- {name} ---")
        print("\n".join(log.read_text().splitlines()[-15:]))
